# PFAS Compliance Screening

Screen battery, semiconductor, and coatings BOMs for PFAS substances.
Visualize urgency levels and replacement candidates.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd

output_dir = '../showcase/outputs'
roadmap_path = os.path.join(output_dir, 'pfas_replacement_roadmap.csv')

if not os.path.exists(roadmap_path):
    print('Generating data...')
    from showcase.pfas_screening import main
    main()

# Load BOM results
bom_files = {
    'Battery': 'pfas_li_ion_battery_cell.json',
    'Semiconductor': 'pfas_semiconductor_fab.json',
    'Coatings': 'pfas_industrial_coatings.json',
}

bom_data = {}
for label, fname in bom_files.items():
    path = os.path.join(output_dir, fname)
    if os.path.exists(path):
        with open(path) as f:
            bom_data[label] = json.load(f)

roadmap = pd.read_csv(roadmap_path)
print(f'BOMs loaded: {list(bom_data.keys())}')
print(f'Replacement roadmap entries: {len(roadmap)}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# BOM Overview: PFAS count by industry
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

urgency_colors = {
    'critical': '#d32f2f', 'high': '#ff5722',
    'moderate': '#ff9800', 'low': '#ffc107', 'none': '#4caf50'
}

for ax, (label, bom) in zip(axes, bom_data.items()):
    pfas_names = [p['material_name'] for p in bom['pfas_materials']]
    clean_count = len(bom['clean_materials'])
    pfas_count = bom['pfas_count']

    ax.bar(['PFAS', 'Clean'], [pfas_count, clean_count],
           color=['#ff5722', '#4caf50'], edgecolor='black')
    ax.set_title(f"{label}\n({bom['total_materials']} materials)")
    ax.set_ylabel('Count')
    if pfas_names:
        ax.annotate('\n'.join(pfas_names), xy=(0, pfas_count),
                    fontsize=8, ha='center', va='bottom')

plt.suptitle('PFAS Exposure by Industry', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Replacement Roadmap Table
print('PFAS Replacement Roadmap')
print('=' * 80)
display_cols = ['pfas_material', 'use_case', 'urgency', 'best_replacement',
                'replacement_score', 'num_alternatives']
roadmap[display_cols]

In [ ]:
# Replacement scores visualization
valid = roadmap.dropna(subset=['replacement_score']).copy()
if not valid.empty:
    valid['label'] = valid['pfas_material'] + ' (' + valid['use_case'] + ')'
    valid = valid.sort_values('replacement_score', ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(4, len(valid) * 0.5)))
    colors = [urgency_colors.get(u, '#999') for u in valid['urgency']]
    ax.barh(valid['label'], valid['replacement_score'],
            color=colors, edgecolor='black', alpha=0.8)

    for i, (score, repl) in enumerate(zip(valid['replacement_score'],
                                           valid['best_replacement'])):
        ax.text(score + 0.01, i, f'{repl} ({score:.2f})', va='center', fontsize=8)

    ax.set_xlabel('Replacement Score')
    ax.set_title('Best PFAS-Free Replacement by Use Case')
    ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print('No valid replacement scores to plot.')

In [ ]:
# PVDF -> CMC+SBR transition detail
print('PVDF Battery Binder Transition Detail')
print('=' * 60)
pvdf_rows = roadmap[roadmap['pfas_material'] == 'PVDF']
for _, row in pvdf_rows.iterrows():
    print(f"Use case: {row['use_case']}")
    print(f"Best replacement: {row['best_replacement']} (score: {row['replacement_score']})")
    print(f"Advantages: {row['replacement_advantages']}")
    print(f"Limitations: {row['replacement_limitations']}")
    print(f"Other alternatives: {row['num_alternatives']}")
    print()